- https://www.zhihu.com/question/15481741792/answer/1983204871277672247

In [2]:
from IPython.display import Image

In [3]:
Image(url='./figs/dp-rl.webp', width=500)

$$
Q^{\pi_{t+1}}(s, a) = \sum_{s'} P(s'|s, a)\left(R(s, a, s') + \gamma \max_{a} Q^{\pi_t}(s', a)\right)
$$

- 在动态规划中，这些环境模型参数是已知量，可以直接代入公式进行精确的数学期望计算，从而推导出最优策略。
    - DP 能直接用 $P$ 和 $R$ 把“期望”算出来，不用去试走每一条路来估计。
    - dsa（Data Structures and Algorithms）两个核心的充要条件：最优子结构（Optimal Substructure）与重叠子问题（Overlapping Subproblems）
        - $V^*(s) = \max_a \sum_{s'} P(s'|s, a) \left( R(s, a, s') + \gamma V^*(s') \right)$
            - 确定性转移时：$V^*(s) = \max_a \left( R(s, a) + \gamma V^*(s') \right)$
        - 要找到节点 A 到终点 F 的最短路，算法根本不需要一次性穷举所有路线。它只需要查看 A 的所有相邻节点，计算“A 到邻居的距离 + 邻居到 F 的最短距离”，并从中挑出一个最小值。贝尔曼方程保证了这种“局部最优组合等于全局最优”的绝对正确性。
        - 仅仅能被拆分是不够的，如果子问题互相独立，那叫分治算法（如归并排序）。动态规划的真正灵魂在于，这些被拆分出来的子问题会在计算过程中被高频地反复遇到。
        - 状态表（DP Table）的引入： 为了打破这种计算冗余，贝尔曼方程在工程实现上天然要求引入**记忆化（Memoization）或表格化（Tabulation）**的数据结构。在代码中，这通常体现为一个一维数组、二维矩阵或哈希映射（如 `V[s]` 或 `Q[s, a]`）。当算法首次算出状态 $s$ 的价值后，便将其沉淀在这块内存空间中。当其他路径再次游走到该状态时，算法直接以 $O(1)$ 的时间复杂度“查表”读取。
- 强化学习本质上是在环境模型（MDP）未知的情况下，通过与环境不断试错采样，来求解动态规划问题的技术。
    - $\gamma \in [0,1)$：折扣因子，越接近 1 越重视长期回报
    - 仍然有状态（或可观测到的表征）
    - 但不知道“从这里做某个动作会去哪、回报多少”（即 $P$ 和 $R$ 不可直接获得）
    - 只能靠 samples（样本）：通过与环境交互得到 $(s,a,r,s')$ 这样的经验
- 进一步延伸 RL 的不同分类
    - 模型驱动（model-based）RL：先学一个近似的 $P,R$，再用DP/规划做推演
        - MPC
        - AlphaGo
    - 无模型（model-free）RL：不显式学 $P, R$ 直接学 $Q$ 或策略 $\pi(a|s)$
        - 价值学习（Value-based）：直接学 $Q/V$
        - 策略梯度（Policy gradient）：直接学 $\pi(a|s)$
        - Actor–Critic：策略（Actor）+ 价值函数（Critic）
- AlphaGo
    - Model-free 侧面： 它的策略网络（Policy Network）和价值网络（Value Network）最初是通过大量人类专家棋谱进行监督学习，随后利用无模型强化学习（自我对弈）来不断更新网络权重。在这个网络评估阶段，模型只是在拟合一种“直觉”，并不涉及对环境未来状态的显式推演。
        - 策略网络 $p(a|s)$: $p_\sigma(a|s)$ (sl), $p_\rho(a|s)$ (rl)
            - $\Delta \sigma \propto \frac{\partial \log p_\sigma(a|s)}{\partial \sigma}$
            - $\Delta \rho \propto \frac{\partial \log p_\rho(a|s_t)}{\partial \rho} z$
        - 价值网络 $v_\theta(s)$: 它扫一眼棋盘 $s$，直接输出一个标量胜率预测。其参数 $\theta$ 通过最小化真实胜负结果 $z$ 与预测值之间的均方误差（MSE）来进行梯度下降：
            - $\Delta \theta \propto \frac{\partial v_\theta(s)}{\partial \theta}(z - v_\theta(s))$
    - Model-based 侧面： 在实际落子的决策阶段，AlphaGo 高度依赖蒙特卡洛树搜索（MCTS）。MCTS 的有效运行建立在一个绝对前提之上——它被硬编码了围棋的完美物理规则（即完美的环境动态模型）。它利用这个预先给定的绝对模型在内部向前推演数万步。因此，就其核心的规划能力而言，AlphaGo 属于依赖已知模型的 Model-based 系统。
        - $s_{t+1} = \mathcal{T}(s_t, a_t)$: 完美的围棋规则引擎 $\mathcal{T}$
        - 选择（Selection）与 PUCT 算法
            - $a_t = \mathop{\arg\max}_{a} \left( Q(s_t, a) + u(s_t, a) \right)$
                - $u(s, a) = c_{\text{puct}} P(s, a) \frac{\sqrt{\sum_{b} N(s, b)}}{1 + N(s, a)}$
        - 评估（Evaluation）的兜底机制
            - 当推演触及树的叶子节点 $s_L$ 时，模型不再利用 $\mathcal{T}$ 继续展开，而是利用价值网络 $v_\theta(s_L)$ 和快速走子策略 $p_\pi$ 的模拟结果 $z_L$ 进行混合评估：
                - $V(s_L) = (1-\lambda) v_\theta(s_L) + \lambda z_L$

### examples

- 状态 ($s$ 和 $s'$)： 图中的圆圈节点（A, B, C, D, E, F）就是状态。
- 动作 ($a$)： 沿着连线从一个节点移动到相邻节点就是动作（例如在状态 A，可以选择动作“去 B”或“去 C”）。
- 奖励 ($R(s, a, s')$)： 连线上的数字（3, 4, 5...）代表成本或代价。在使用要求最大化的贝尔曼方程求解时，我们通常将这些代价转化为负奖励（例如走代价为 3 的路，获得 -3 的 Reward）。
- 状态转移概率 ($P(s'|s, a)$)： 在左图这种确定的图结构中，如果你在 A 点决定走向 B 点，你 100% 会到达 B 点。因此，$P(s'|s, a)$ 在这里的取值为 1（对于目标节点）或 0（对于其他无关节点）。
    - 由于 $P=1$，原方程中的求和符号 $\sum$ 其实就去掉了，方程直接简化为：$$Q(s, a) = R(s, a, s') + \gamma \max_{a'} Q(s', a')$$

In [1]:
import numpy as np

def solve_dp_white_box():
    # 1. 定义“白盒”环境模型 (MDP 的已知信息)
    # 图中的节点为状态 (States)，连线为动作 (Actions)，数字为代价 (Costs)
    # 为了使用带 max 的贝尔曼方程，将正数代价转化为负数奖励 (Rewards)
    graph = {
        'A': {'B': 3, 'C': 5, 'D': 9},
        'B': {'A': 3, 'C': 3, 'D': 4, 'E': 7},
        'C': {'A': 5, 'B': 3, 'D': 2, 'E': 6, 'F': 8},
        'D': {'A': 9, 'B': 4, 'C': 2, 'E': 2, 'F': 2},
        'E': {'B': 7, 'C': 6, 'D': 2, 'F': 5},
        'F': {}  # F 为吸收状态（终点）
    }

    states = list(graph.keys())
    
    # 2. 初始化价值函数 V(s) 和 动作价值 Q(s, a)
    # 终点 F 的价值为 0，其余状态初始化为负无穷大
    V = {s: -float('inf') for s in states}
    V['F'] = 0 
    Q = {s: {} for s in states if s != 'F'}

    gamma = 1.0       # 折扣因子 (最短路径问题通常设为 1)
    threshold = 1e-5  # 迭代收敛阈值

    # 3. 核心：应用图片左下角的贝尔曼方程进行价值迭代
    iteration = 0
    while True:
        delta = 0
        V_new = V.copy()

        for s in states:
            if s == 'F': 
                continue

            # 遍历当前状态 s 能采取的所有动作（即走向相邻状态 s_next）
            for s_next, cost in graph[s].items():
                
                # 对应图中红框部分：环境模型完全已知
                # P(s'|s,a) = 1 （确定性转移）
                # R(s,a,s') = -cost （代价即负奖励）
                reward = -cost 
                
                # 核心公式：Q(s, a) = R(s, a, s') + gamma * max_a' Q(s', a')
                # 其中 max_a' Q(s', a') 在数学上等同于下一状态的 V(s_next)
                Q[s][s_next] = reward + gamma * V[s_next]

            # 更新当前状态的价值：寻找能带来最大 Q 值的动作
            best_value = max(Q[s].values())
            delta = max(delta, abs(V[s] - best_value))
            V_new[s] = best_value

        V = V_new
        iteration += 1
        
        # 检查是否收敛
        if delta < threshold:
            break

    # 4. 根据收敛后的 Q 表，提取最优策略 (Policy)
    policy = {}
    for s in states:
        if s == 'F': 
            continue
        # 找出让 Q 值最大的动作 (即走向哪个邻居)
        best_action = max(Q[s], key=Q[s].get)
        policy[s] = best_action

    return iteration, V, policy

# 运行求解并打印洞察
iteration, V, policy = solve_dp_white_box()

print(f"动态规划在第 {iteration} 次迭代后完成收敛。\n")

print("--- 状态价值 V(s) [即到达终点的最小累计代价的负值] ---")
for s, val in V.items():
    print(f"状态 {s}: {val}")

print("\n--- 全局最优策略 (上帝视角下的确定性路线) ---")
for s, a in policy.items():
    print(f"在节点 {s}，最优动作是走向 -> {a}")

print("\n--- 实际推演: 从 A 到 F 的最优路径 ---")
path = ['A']
current = 'A'
while current != 'F':
    current = policy[current]
    path.append(current)
print(" -> ".join(path))

动态规划在第 4 次迭代后完成收敛。

--- 状态价值 V(s) [即到达终点的最小累计代价的负值] ---
状态 A: -9.0
状态 B: -6.0
状态 C: -4.0
状态 D: -2.0
状态 E: -4.0
状态 F: 0

--- 全局最优策略 (上帝视角下的确定性路线) ---
在节点 A，最优动作是走向 -> B
在节点 B，最优动作是走向 -> D
在节点 C，最优动作是走向 -> D
在节点 D，最优动作是走向 -> F
在节点 E，最优动作是走向 -> D

--- 实际推演: 从 A 到 F 的最优路径 ---
A -> B -> D -> F


$$
dp[v] = \min(dp[v], dp[u] + w)
$$
- 一个经典的**单源最短路径（Single-Source Shortest Path, SSSP）**问题。
- DSA 视角的 DP（Bellman-Ford）
    - 状态定义 (State)： `dp[u]` 表示从起点 A 到达节点 u 的最小总代价（最短距离）。
    - 基本情况 (Base Case)： `dp['A'] = 0`（起点到自己代价为 0），其余节点的 dp 值初始化为正无穷大 +∞。
    - 状态转移方程 (State Transition / Relaxation)： 对于图中的任意一条边 (u, v)，其权重为 w。
    - 到达节点 v 的最短距离，要么保持原样，要么是先以最短距离到达节点 u，再从 u 走过这条边到达 v

In [4]:
def solve_shortest_path_dsa_dp():
    # 1. 定义图的边集合 (Edge List)
    # DSA 中通常将边表示为 (起点, 终点, 权重) 的元组
    # 图中没有箭头，我们将其视为无向图，因此每条边需要双向添加
    edges_raw = [
        ('A', 'B', 3), ('A', 'C', 5), ('A', 'D', 9),
        ('B', 'C', 3), ('B', 'D', 4), ('B', 'E', 7),
        ('C', 'D', 2), ('C', 'E', 6), ('C', 'F', 8),
        ('D', 'E', 2), ('D', 'F', 2),
        ('E', 'F', 5)
    ]
    
    edges = []
    for u, v, w in edges_raw:
        edges.append((u, v, w))
        edges.append((v, u, w)) # 无向图双向连通

    nodes = ['A', 'B', 'C', 'D', 'E', 'F']
    
    # 2. 初始化 DP 状态表 (1D DP Array)
    dp = {node: float('inf') for node in nodes}
    dp['A'] = 0  # Base case: 起点代价为 0
    
    # 用于记录最优路径的“前驱节点”，方便最后回溯
    parent = {node: None for node in nodes}
    
    # 3. 核心：DP 状态转移 (Bellman-Ford 算法的主循环)
    # 理论基础：在一个包含 V 个节点的图中，一条最短路径最多只包含 V-1 条边
    for iteration in range(len(nodes) - 1):
        updated = False  # 提前剪枝优化标志
        
        # 遍历所有边，尝试“松弛(Relax)”每一个节点的最短距离
        for u, v, w in edges:
            # 状态转移方程：如果经过 u 到达 v 更近，则更新 dp[v]
            if dp[u] + w < dp[v]:
                dp[v] = dp[u] + w
                parent[v] = u
                updated = True
        
        # 如果在某次遍历中没有任何 dp 值被更新，说明已经找到了全局最优解，提前退出
        if not updated:
            print(f"DSA 算法在第 {iteration + 1} 轮遍历后完成收敛。")
            break

    # 4. 回溯提取最短路径 (从终点 F 倒推回起点 A)
    path = []
    curr = 'F'
    while curr is not None:
        path.append(curr)
        curr = parent[curr]
    path.reverse()  # 将路径翻转为从前往后的顺序
    
    return dp, path

# 运行算法并输出结果
dp_table, best_path = solve_shortest_path_dsa_dp()

print("\n--- DSA 动态规划：最终的 DP 状态表 ---")
for node, cost in dp_table.items():
    print(f"起点 A 到节点 {node} 的最短距离: dp[{node}] = {cost}")

print(f"\n--- 回溯得到的最优路径 ---")
print(" -> ".join(best_path), f" (总代价: {dp_table['F']})")

DSA 算法在第 2 轮遍历后完成收敛。

--- DSA 动态规划：最终的 DP 状态表 ---
起点 A 到节点 A 的最短距离: dp[A] = 0
起点 A 到节点 B 的最短距离: dp[B] = 3
起点 A 到节点 C 的最短距离: dp[C] = 5
起点 A 到节点 D 的最短距离: dp[D] = 7
起点 A 到节点 E 的最短距离: dp[E] = 9
起点 A 到节点 F 的最短距离: dp[F] = 9

--- 回溯得到的最优路径 ---
A -> B -> D -> F  (总代价: 9)
